# Parcial de Big Data · Clasificación multiclase con Spark ML

El objetivo es estimar, al cierre del segundo semestre, si un estudiante terminará en una de tres clases: **Dropout** (deserción), **Enrolled** (aún matriculado) o **Graduate** (graduación). Se comparan dos modelos: regresión logística multinomial y Random Forest.

En Google Colab, abre el archivo y elige **Entorno de ejecución → Ejecutar todas** con un entorno de CPU. Al final se genera `modelo_final.zip`, que incluye el pipeline y los metadatos para el formulario y la lectura de probabilidades.

## A. Carga, limpieza, ingeniería de características y partición

### Entorno de Spark

Instalamos una versión fija de PySpark para poder recargar el modelo con la misma versión. Limitamos Spark a dos hilos y los *shuffles* a cuatro particiones como máximo. Usamos los 4 GB de memoria del driver que recomienda el enunciado.

In [1]:
import os
import subprocess
import sys

EN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_JUPYTER_IP" in os.environ
SPARK_VERSION = "3.5.8"
if EN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={SPARK_VERSION}"],
        check=True,
    )

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.master("local[2]")
    .appName("dropout_sparkml")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.sql.constraintPropagation.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "2")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | {spark.sparkContext.master}")

Spark 3.5.8 | local[2]


### Carga y distribución de clases

Leemos con Spark el CSV con encabezado, tipos inferidos y separador de punto y coma. Si existe `data.csv` en el entorno, se usa ese archivo. En caso contrario se intenta descargar la copia de Kaggle indicada en el enunciado y, si no se puede obtener el CSV esperado, se usa la publicación original. El esquema, las dimensiones y los conteos permiten verificar que se cargó el conjunto correcto. [Ficha del conjunto en UCI](https://doi.org/10.24432/C5MC89)

In [2]:
import io
import re
from pathlib import Path
from urllib.request import urlopen
from zipfile import BadZipFile, ZipFile

KAGGLE = "https://www.kaggle.com/api/v1/datasets/download/aldishamim8/predict-students-dropout-and-academic-success"
UCI = "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip"
csv_path = Path("data.csv")

if csv_path.exists():
    fuente = os.environ.get("FUENTE_CSV", f"Archivo local {csv_path.name}")
else:
    for url, fuente in [(KAGGLE, "Kaggle"), (UCI, "UCI Machine Learning Repository")]:
        try:
            with urlopen(url, timeout=20) as response:
                with ZipFile(io.BytesIO(response.read())) as archive:
                    for name in archive.namelist():
                        if name.lower().endswith(".csv"):
                            data = archive.read(name)
                            header = data.splitlines()[0].decode("utf-8-sig").lower()
                            if header.endswith(";target") and "curricular units" in header:
                                csv_path.write_bytes(data)
                                break
        except (OSError, ValueError, BadZipFile) as error:
            print(f"{fuente}: {error}")
        if csv_path.exists():
            break
    if not csv_path.exists():
        raise FileNotFoundError("Sube el CSV original separado por ';' con el nombre data.csv.")

raw = spark.read.csv(csv_path.resolve().as_uri(), header=True, inferSchema=True, sep=";")
if "Target" not in raw.columns:
    raise ValueError("Se esperaba una columna Target; verifica el archivo y el separador.")
raw.printSchema()
total = raw.count()
print(f"Fuente: {fuente} | Filas: {total} | Columnas: {len(raw.columns)}")
distribucion = raw.groupBy("Target").count().orderBy(F.desc("count"))
distribucion.show()

root
 |-- Marital status: integer (nullable = true)
 |-- Application mode: integer (nullable = true)
 |-- Application order: integer (nullable = true)
 |-- Course: integer (nullable = true)
 |-- Daytime/evening attendance\t: integer (nullable = true)
 |-- Previous qualification: integer (nullable = true)
 |-- Previous qualification (grade): double (nullable = true)
 |-- Nacionality: integer (nullable = true)
 |-- Mother's qualification: integer (nullable = true)
 |-- Father's qualification: integer (nullable = true)
 |-- Mother's occupation: integer (nullable = true)
 |-- Father's occupation: integer (nullable = true)
 |-- Admission grade: double (nullable = true)
 |-- Displaced: integer (nullable = true)
 |-- Educational special needs: integer (nullable = true)
 |-- Debtor: integer (nullable = true)
 |-- Tuition fees up to date: integer (nullable = true)
 |-- Gender: integer (nullable = true)
 |-- Scholarship holder: integer (nullable = true)
 |-- Age at enrollment: integer (nullable 

**Desbalance.** En el conjunto original, **Graduate** domina con 2.209 estudiantes (49,93 %), **Dropout** tiene 1.421 (32,12 %) y **Enrolled** es la minoritaria con 794 (17,95 %). Un modelo que siempre eligiera Graduate acertaría casi la mitad de los casos y dejaría sin identificar a las otras dos clases. El **F1 ponderado** combina precisión y recall por clase y pondera cada F1 por su número de casos. Mantendremos la distribución observada, sin remuestreo ni pesos adicionales, y examinaremos el recall y la matriz de confusión de la clase minoritaria porque el F1 ponderado da más peso a las clases frecuentes.

### Limpieza y verificación

Uniformamos los nombres con una expresión regular, convertimos cadenas vacías en nulos y quitamos filas incompletas o duplicadas. Verificamos que las tres etiquetas y los tipos de los datos estén disponibles antes de definir las entradas.

In [3]:
from pyspark.sql.types import NumericType, StringType

nombres = [re.sub(r"[^a-z0-9]+", "", col.lower()) for col in raw.columns]
if len(set(nombres)) != len(nombres):
    raise ValueError("La normalización generó nombres repetidos.")
datos = raw.toDF(*nombres)
for field in datos.schema:
    if isinstance(field.dataType, StringType):
        trimmed = F.trim(F.col(field.name))
        datos = datos.withColumn(field.name, F.when(trimmed != "", trimmed))

sin_nulos = datos.dropna()
n_completos = sin_nulos.count()
limpios = sin_nulos.dropDuplicates().coalesce(2).cache()
n_limpios = limpios.count()
print(f"Filas incompletas: {total - n_completos}")
print(f"Duplicados entre filas completas: {n_completos - n_limpios}")
print(f"Filas válidas: {n_limpios}")
clases = {row.target for row in limpios.select("target").distinct().collect()}
assert clases == {"Dropout", "Enrolled", "Graduate"}, clases
display({field.name: field.dataType.simpleString() for field in limpios.schema})

Filas incompletas: 0
Duplicados entre filas completas: 0
Filas válidas: 4424
{'maritalstatus': 'int', 'applicationmode': 'int', 'applicationorder': 'int', 'course': 'int', 'daytimeeveningattendance': 'int', 'previousqualification': 'int', 'previousqualificationgrade': 'double', 'nacionality': 'int', 'mothersqualification': 'int', 'fathersqualification': 'int', 'mothersoccupation': 'int', 'fathersoccupation': 'int', 'admissiongrade': 'double', 'displaced': 'int', 'educationalspecialneeds': 'int', 'debtor': 'int', 'tuitionfeesuptodate': 'int', 'gender': 'int', 'scholarshipholder': 'int', 'ageatenrollment': 'int', 'international': 'int', 'curricularunits1stsemcredited': 'int', 'curricularunits1stsemenrolled': 'int', 'curricularunits1stsemevaluations': 'int', 'curricularunits1stsemapproved': 'int', 'curricularunits1stsemgrade': 'double', 'curricularunits1stsemwithoutevaluations': 'int', 'curricularunits2ndsemcredited': 'int', 'curricularunits2ndsemenrolled': 'int', 'curricularunits2ndsemev

### Variables de entrada y variables derivadas

Seleccionamos entradas por pertinencia académica y disponibilidad al cierre del segundo semestre. Programa, jornada y tipo de estudios previos son categorías codificadas como números. Las notas, unidades y conteos son variables numéricas. Priorizamos el historial académico y dejamos fuera los datos demográficos y macroeconómicos. En la sección C revisamos las importancias de Random Forest como diagnóstico del aporte de las variables seleccionadas.

- **Tasa de aprobación del primer semestre.** Unidades aprobadas divididas por unidades matriculadas en ese semestre.
- **Tasa de aprobación del segundo semestre.** La misma proporción en el segundo semestre. Ambas tasas se fijan en cero si la matrícula es cero, y los conteos de unidades quedan disponibles para distinguir esa situación.
- **Variación de la nota.** Nota del segundo semestre menos nota del primero. Si falta actividad representada por una nota positiva en alguno de los dos semestres, se fija la variación en cero y se conservan las notas originales.

Un `SQLTransformer` calcula estas variables dentro del pipeline guardado. Así se aplican las mismas fórmulas en entrenamiento y al predecir desde la app.

In [4]:
from pyspark.ml.feature import SQLTransformer

categoricas = ["course", "daytimeeveningattendance", "previousqualification"]
numericas_base = ["admissiongrade", "previousqualificationgrade"] + [
    f"curricularunits{sem}sem{medida}"
    for sem in ("1st", "2nd")
    for medida in ("enrolled", "approved", "grade", "evaluations")
]
entradas = categoricas + numericas_base
faltantes = sorted(set(entradas) - set(limpios.columns))
if faltantes:
    raise ValueError(f"Faltan entradas requeridas en el CSV: {faltantes}")
tipos_no_numericos = [
    col for col in entradas if not isinstance(limpios.schema[col].dataType, NumericType)
]
if tipos_no_numericos:
    raise TypeError(f"Se esperaban códigos o valores numéricos: {tipos_no_numericos}")

derivadas = ["tasaaprobacion1", "tasaaprobacion2", "variacionnota"]
numericas = numericas_base + derivadas
sql_derivadas = '''
    SELECT *,
      CASE WHEN curricularunits1stsemenrolled > 0
           THEN curricularunits1stsemapproved / curricularunits1stsemenrolled
           ELSE 0.0 END AS tasaaprobacion1,
      CASE WHEN curricularunits2ndsemenrolled > 0
           THEN curricularunits2ndsemapproved / curricularunits2ndsemenrolled
           ELSE 0.0 END AS tasaaprobacion2,
      CASE WHEN curricularunits1stsemgrade > 0 AND curricularunits2ndsemgrade > 0
           THEN curricularunits2ndsemgrade - curricularunits1stsemgrade
           ELSE 0.0 END AS variacionnota
    FROM __THIS__
'''
SQLTransformer(statement=sql_derivadas).transform(limpios).select(
    *derivadas
).show(5)

+---------------+------------------+-------------------+
|tasaaprobacion1|   tasaaprobacion2|      variacionnota|
+---------------+------------------+-------------------+
|            0.0|               0.0|                0.0|
|            1.0|0.8333333333333334|-1.0285714285714285|
|            1.0|               1.0|              1.045|
|            0.0|               0.0|                0.0|
|            1.0|0.8333333333333334| 0.4285714285714288|
+---------------+------------------+-------------------+
only showing top 5 rows



### Partición e índice de las etiquetas

Usamos `randomSplit` con proporciones 80/20 y semilla 42. Ajustamos el índice de `Target` fuera del pipeline y solo en entrenamiento, y aplicamos ese mismo índice a prueba. El orden de `labels` se guarda para que las probabilidades de la app correspondan a las clases correctas.

In [5]:
from pyspark.ml.feature import StringIndexer

SEMILLA = 42
base = limpios.select("target", *entradas).cache()
base.count()
entrenamiento_raw, prueba_raw = base.randomSplit([0.8, 0.2], seed=SEMILLA)
indexador_target = StringIndexer(
    inputCol="target", outputCol="label", stringOrderType="frequencyDesc"
).fit(entrenamiento_raw)
labels = indexador_target.labels
assert set(labels) == clases
entrenamiento = indexador_target.transform(entrenamiento_raw).cache()
prueba = indexador_target.transform(prueba_raw).cache()
print(f"Entrenamiento: {entrenamiento.count()} | Prueba: {prueba.count()}")
print("Índice → clase:", dict(enumerate(labels)))
for nombre, datos_split in [("Entrenamiento", entrenamiento), ("Prueba", prueba)]:
    print(nombre)
    datos_split.groupBy("target").count().orderBy(F.desc("count")).show()

Entrenamiento: 3595 | Prueba: 829
Índice → clase: {0: 'Graduate', 1: 'Dropout', 2: 'Enrolled'}
Entrenamiento
+--------+-----+
|  target|count|
+--------+-----+
|Graduate| 1780|
| Dropout| 1159|
|Enrolled|  656|
+--------+-----+

Prueba
+--------+-----+
|  target|count|
+--------+-----+
|Graduate|  429|
| Dropout|  262|
|Enrolled|  138|
+--------+-----+



## B. Pipelines, hiperparametrización y validación cruzada

### Preprocesamiento compartido

Cada pipeline calcula las variables derivadas e indexa las categorías con `handleInvalid="keep"` para admitir niveles nuevos. `OneHotEncoder` evita imponer un orden a los códigos. `VectorAssembler` combina las entradas y `StandardScaler` ajusta sus escalas sin centrar el vector. La escala ayuda a la logística con la regularización. Los árboles se ajustan con la misma representación para mantener el preprocesamiento requerido.

In [6]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder


def construir_pipeline(clasificador):
    indices = [f"{col}_idx" for col in categoricas]
    indicadores = [f"{col}_ohe" for col in categoricas]
    return Pipeline(stages=[
        SQLTransformer(statement=sql_derivadas),
        StringIndexer(inputCols=categoricas, outputCols=indices, handleInvalid="keep"),
        OneHotEncoder(inputCols=indices, outputCols=indicadores, dropLast=False),
        VectorAssembler(inputCols=numericas + indicadores, outputCol="vector"),
        StandardScaler(inputCol="vector", outputCol="features", withMean=False),
        clasificador,
    ])

### Búsqueda de hiperparámetros

Probamos cuatro combinaciones por algoritmo con `ParamGridBuilder` y tres pliegues con `CrossValidator`. La regresión logística ajusta la fuerza de regularización (`regParam`) y la mezcla L1/L2 (`elasticNetParam`). Random Forest ajusta el número de árboles (`numTrees`) y su profundidad máxima (`maxDepth`). Usamos la misma semilla para la asignación de pliegues y una combinación a la vez para limitar la memoria. El preprocesamiento se ajusta dentro de cada pliegue.

In [7]:
logistica = LogisticRegression(
    family="multinomial", maxIter=50, standardization=False
)
bosque = RandomForestClassifier(seed=SEMILLA)
grilla_logistica = (
    ParamGridBuilder()
    .addGrid(logistica.regParam, [0.01, 0.1])
    .addGrid(logistica.elasticNetParam, [0.0, 0.5])
    .build()
)
grilla_bosque = (
    ParamGridBuilder()
    .addGrid(bosque.numTrees, [30, 60])
    .addGrid(bosque.maxDepth, [5, 8])
    .build()
)
configuraciones = {
    "Regresión logística": (logistica, grilla_logistica),
    "Random Forest": (bosque, grilla_bosque),
}
evaluador = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
ajustes, resultados_cv, detalle_cv = {}, {}, {}

for nombre, (clasificador, grilla) in configuraciones.items():
    print(f"Entrenando {nombre}…", flush=True)
    ajuste = CrossValidator(
        estimator=construir_pipeline(clasificador),
        estimatorParamMaps=grilla,
        evaluator=evaluador,
        numFolds=3,
        seed=SEMILLA,
        parallelism=1,
    ).fit(entrenamiento)
    ganador = max(range(len(grilla)), key=lambda i: ajuste.avgMetrics[i])
    ajustes[nombre] = ajuste
    resultados_cv[nombre] = ajuste.avgMetrics[ganador]
    detalle_cv[nombre] = {param.name: valor for param, valor in grilla[ganador].items()}
    print(f"Mejor combinación: {detalle_cv[nombre]} | F1 CV: {resultados_cv[nombre]:.4f}")
    for parametros, f1 in zip(grilla, ajuste.avgMetrics):
        print({param.name: valor for param, valor in parametros.items()}, f"F1: {f1:.4f}")

Entrenando Regresión logística…
Mejor combinación: {'regParam': 0.01, 'elasticNetParam': 0.0} | F1 CV: 0.7203
{'regParam': 0.01, 'elasticNetParam': 0.0} F1: 0.7203
{'regParam': 0.01, 'elasticNetParam': 0.5} F1: 0.7094
{'regParam': 0.1, 'elasticNetParam': 0.0} F1: 0.6948
{'regParam': 0.1, 'elasticNetParam': 0.5} F1: 0.6396
Entrenando Random Forest…
Mejor combinación: {'numTrees': 60, 'maxDepth': 8} | F1 CV: 0.7078
{'numTrees': 30, 'maxDepth': 5} F1: 0.6905
{'numTrees': 30, 'maxDepth': 8} F1: 0.7063
{'numTrees': 60, 'maxDepth': 5} F1: 0.6864
{'numTrees': 60, 'maxDepth': 8} F1: 0.7078


### Interpretación de los ajustes

La regresión logística logró su mayor F1 de validación (0.7203) con `regParam=0.01` y `elasticNetParam=0.0`. La penalización L2 dio el mejor resultado entre las opciones probadas. Random Forest obtuvo 0.7078 con 60 árboles y profundidad máxima de 8. La profundidad controla la complejidad de las reglas y el número de árboles controla cuántos árboles participan en la predicción. Estos resultados corresponden a los valores de la cuadrícula evaluada.

## C. Evaluación, selección y guardado del mejor modelo

### Comparación sobre el conjunto de prueba

El mayor F1 medio de validación cruzada determina el pipeline seleccionado. Se extrae mediante `bestModel`. Calculamos las cuatro métricas para ambos modelos sobre las filas reservadas para prueba y comparamos también con un clasificador de referencia que siempre elegiría la clase más frecuente en entrenamiento.

In [8]:
nombre_final = max(resultados_cv, key=resultados_cv.get)
modelo_final = ajustes[nombre_final].bestModel
metricas = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision_ponderada": "weightedPrecision",
    "recall_ponderado": "weightedRecall",
}
comparacion = []
for nombre, ajuste in ajustes.items():
    pred = ajuste.bestModel.transform(prueba).select("label", "prediction").cache()
    valores = {
        etiqueta: evaluador.evaluate(pred, {evaluador.metricName: metrica})
        for etiqueta, metrica in metricas.items()
    }
    comparacion.append({"modelo": nombre, "f1_cv": resultados_cv[nombre], **valores})
    pred.unpersist()

print(f"{'Modelo':<22} | {'F1 CV':>7} | {'Accuracy':>8} | {'F1 prueba':>9} | "
      f"{'Precisión pond.':>15} | {'Recall pond.':>12}")
for fila in sorted(comparacion, key=lambda r: -r["f1_cv"]):
    print(f"{fila['modelo']:<22} | {fila['f1_cv']:>7.4f} | {fila['accuracy']:>8.4f} | "
          f"{fila['f1']:>9.4f} | {fila['precision_ponderada']:>15.4f} | "
          f"{fila['recall_ponderado']:>12.4f}")
print(f"Seleccionado por F1 de validación cruzada: {nombre_final}")
print(f"Accuracy de referencia ({labels[0]}): {prueba.filter(F.col('label') == 0).count() / prueba.count():.4f}")

Modelo                 |   F1 CV | Accuracy | F1 prueba | Precisión pond. | Recall pond.
Regresión logística    |  0.7203 |   0.7322 |    0.7169 |          0.7119 |       0.7322
Random Forest          |  0.7078 |   0.7575 |    0.7413 |          0.7424 |       0.7575
Seleccionado por F1 de validación cruzada: Regresión logística
Accuracy de referencia (Graduate): 0.5175


### Confusión por clase e importancia de variables

La matriz del modelo seleccionado muestra la clase real en las filas y la predicción en las columnas. El recall de cada clase mide qué parte de sus casos detectó el modelo. En particular, examinamos `Enrolled` por ser la clase minoritaria. Para entender el bosque, agrupamos las importancias de los indicadores de una misma categoría. Estas importancias describen el uso de las variables en los árboles y pueden repartirse entre variables correlacionadas, como las unidades aprobadas y su tasa de aprobación.

In [9]:
pred_final = modelo_final.transform(prueba).select("label", "prediction").cache()
conteos = {
    (int(row.label), int(row.prediction)): row["count"]
    for row in pred_final.groupBy("label", "prediction").count().collect()
}
matriz = [
    [conteos.get((i, j), 0) for j in range(len(labels))]
    for i in range(len(labels))
]
print(f"{'Real / Predicción':<18} | " + " | ".join(f"{clase:>9}" for clase in labels))
for clase, fila in zip(labels, matriz):
    print(f"{clase:<18} | " + " | ".join(f"{cantidad:>9}" for cantidad in fila))

recall_clases = {
    clase: matriz[i][i] / sum(matriz[i]) if sum(matriz[i]) else 0.0
    for i, clase in enumerate(labels)
}
for clase, recall in recall_clases.items():
    print(f"Recall {clase}: {recall:.2%}")

pipeline_bosque = ajustes["Random Forest"].bestModel
encoder = pipeline_bosque.stages[2]
pesos = pipeline_bosque.stages[-1].featureImportances.toArray()
importancias = dict(zip(numericas, map(float, pesos[:len(numericas)])))
inicio = len(numericas)
for col, cantidad in zip(categoricas, encoder.categorySizes):
    importancias[col] = float(pesos[inicio:inicio + cantidad].sum())
    inicio += cantidad
assert inicio == len(pesos)
ranking = sorted(importancias.items(), key=lambda par: -par[1])
print("\nVariables con mayor importancia en Random Forest")
for variable, peso in ranking[:10]:
    print(f"{variable:<35} {peso:.4f}")

Real / Predicción  |  Graduate |   Dropout |  Enrolled
Graduate           |       391 |        24 |        14
Dropout            |        43 |       177 |        42
Enrolled           |        65 |        34 |        39
Recall Graduate: 91.14%
Recall Dropout: 67.56%
Recall Enrolled: 28.26%

Variables con mayor importancia en Random Forest
tasaaprobacion2                     0.2240
curricularunits2ndsemapproved       0.1658
curricularunits1stsemapproved       0.1139
tasaaprobacion1                     0.1114
curricularunits2ndsemgrade          0.0717
curricularunits1stsemgrade          0.0565
course                              0.0530
curricularunits2ndsemevaluations    0.0369
curricularunits1stsemevaluations    0.0360
curricularunits2ndsemenrolled       0.0274


### Selección y resultados

La **regresión logística multinomial** obtuvo el mayor F1 ponderado medio en validación cruzada (**0,7203** frente a **0,7078** de Random Forest). Aplicamos el criterio de selección fijado antes de abrir el conjunto de prueba y guardamos la regresión logística. En las 829 filas de prueba, Random Forest obtuvo el mayor F1 (**0,7413** frente a **0,7169**) y la mayor exactitud (**75,75 %** frente a **73,22 %**). La diferencia entre ambos conjuntos muestra que la clasificación de los modelos varía con las muestras evaluadas.

La regresión logística identificó a **39 de 138** estudiantes de la clase minoritaria Enrolled (28,26 % de recall), frente a 67,56 % para Dropout y 91,14 % para Graduate. El F1 ponderado resume el resultado global, mientras que la matriz permite ver que la predicción de Enrolled sigue siendo la principal limitación del modelo seleccionado. En Random Forest, la tasa de aprobación y el número de unidades aprobadas del segundo semestre fueron las variables con mayor importancia.

### Guardado y comprobación de inferencia

Guardamos el `PipelineModel` completo en `modelo_final`. En la misma carpeta queda `labels.json` con el orden de clases, además de los campos y tipos del formulario. Volvemos a cargar el pipeline, hacemos una predicción con una fila de prueba que contiene solo las entradas del formulario y comprobamos que la clase y las probabilidades coincidan. Empaquetamos la carpeta completa para su posterior uso con Streamlit.

In [10]:
import json
from zipfile import ZIP_DEFLATED

carpeta = Path("modelo_final")
modelo_final.write().overwrite().save("modelo_final")
(carpeta / "labels.json").write_text(
    json.dumps(labels, ensure_ascii=False, indent=2), encoding="utf-8"
)
metadatos = {
    "spark_version": spark.version,
    "modelo": nombre_final,
    "categoricas": categoricas,
    "numericas": numericas_base,
    "tipos": {col: limpios.schema[col].dataType.simpleString() for col in entradas},
}
(carpeta / "entradas.json").write_text(
    json.dumps(metadatos, ensure_ascii=False, indent=2), encoding="utf-8"
)

recargado = PipelineModel.load("modelo_final")
ejemplo = prueba.select(*entradas).limit(1).cache()
original = modelo_final.transform(ejemplo).select("prediction", "probability").first()
recuperado = recargado.transform(ejemplo).select("prediction", "probability").first()
assert original == recuperado
probabilidades = list(map(float, recuperado.probability))
assert len(probabilidades) == len(labels) and abs(sum(probabilidades) - 1.0) < 1e-8
print("Predicción de comprobación:", labels[int(recuperado.prediction)])
for clase, probabilidad in zip(labels, probabilidades):
    print(f"{clase}: {probabilidad:.2%}")

zip_path = Path("modelo_final.zip")
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for archivo in sorted(carpeta.rglob("*")):
        if archivo.is_file() and not archivo.name.endswith(".crc"):
            archive.write(archivo, archivo.as_posix())
print(f"ZIP generado: {zip_path.resolve()} ({zip_path.stat().st_size:,} bytes)")
if EN_COLAB:
    from google.colab import files
    files.download(str(zip_path))

Predicción de comprobación: Dropout
Graduate: 4.83%
Dropout: 58.40%
Enrolled: 36.77%
ZIP generado: /mnt/c/Users/samue/Documents/Codex/2026-11-19/nde/work/uci/modelo_final.zip (12,888 bytes)
